In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
df=pd.read_csv("insurance.csv")

In [3]:
df.head()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
0,67,119.8,1.56,2.92,False,Jaipur,retired,High
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low
2,39,56.8,1.64,36.64,False,Indore,freelancer,Low
3,22,109.4,1.55,3.34,True,Mumbai,student,Medium
4,69,62.2,1.60,3.94,True,Indore,retired,High


In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
5,53,62.9,1.66,50.00,False,Kota,freelancer,Medium
62,34,72.8,1.83,35.67,False,Chennai,business_owner,Low
71,38,54.1,1.81,20.25,False,Chandigarh,unemployed,Low
0,67,119.8,1.56,2.92,False,Jaipur,retired,High
33,73,67.5,1.76,1.46,False,Mumbai,retired,Medium


In [5]:
df['occupation'].unique()

array(['retired', 'freelancer', 'student', 'government_job',
       'business_owner', 'unemployed', 'private_job'], dtype=object)

In [6]:
df_feat=df.copy()

In [7]:
# feature 1:BMI
df_feat['bmi']=df_feat['weight']/(df_feat['height']**2)

In [8]:
#fEATURE 2:age group
def age_group(age):
    if age< 25:
        return 'young'
    elif age<40:
        return 'adult'    
    elif age<60:
        return 'middle age'
    return 'senior'    


In [9]:
df_feat['age_group']=df_feat['age'].apply(age_group)

In [10]:
#feature3:life style risk
def lifestyle_risk(row):
    if row['smoker'] and row['bmi']>30:
        return 'high'
    elif row['smoker'] and row['bmi']>27:
        return 'medium'
    else:
        return 'low'        

In [11]:
df_feat['lifestyle_risk']=df_feat.apply(lifestyle_risk,axis=1)

In [12]:
tier_1_cities = [
    "Mumbai", "Delhi", "Bangalore", "Chennai",
    "Kolkata", "Hyderabad", "Pune"
]

tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi",
    "Visakhapatnam", "Coimbatore", "Bhopal", "Nagpur", "Vadodara",
    "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati",
    "Thiruvananthapuram", "Ludhiana", "Nashik", "Allahabad",
    "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem",
    "Vijayawada", "Tiruchirappalli", "Bhavnagar", "Gwalior",
    "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode",
    "Warangal", "Kolhapur", "Bilaspur", "Jalandhar", "Noida",
    "Guntur", "Asansol", "Siliguri"
]

In [13]:
#feature4: City tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3        

In [14]:
df_feat['city_tier']=df_feat['city'].apply(city_tier)

In [15]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)
     

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
57,1.36,retired,26.889815,senior,low,2,High
54,3.32,retired,21.025423,senior,low,2,High
45,18.39,unemployed,33.466667,middle age,low,2,High
30,32.97,business_owner,29.937519,adult,low,1,Low
9,43.07,business_owner,24.858833,middle age,low,1,Low


In [16]:
# Select features and target
X = df_feat[["bmi", "age_group", "lifestyle_risk", "city_tier", "income_lpa", "occupation"]]
y = df_feat["insurance_premium_category"]

In [17]:
X

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.227482,senior,low,2,2.92000,retired
1,30.189017,adult,low,1,34.28000,freelancer
2,21.118382,adult,low,2,36.64000,freelancer
3,45.535900,young,high,1,3.34000,student
4,24.296875,senior,low,2,3.94000,retired
...,...,...,...,...,...,...
95,21.420747,adult,low,2,19.64000,business_owner
96,47.984483,adult,low,1,34.01000,private_job
97,18.765432,middle age,low,1,44.86000,freelancer
98,30.521676,adult,low,1,28.30000,business_owner


In [18]:
y

0       High
1        Low
2        Low
3     Medium
4       High
       ...  
95       Low
96       Low
97       Low
98       Low
99       Low
Name: insurance_premium_category, Length: 100, dtype: object

In [19]:
#define categorical and numerical features
categorical_features=['age_group','lifestyle_risk','occupation','city_tier']
numerical_features=['bmi','income_lpa']

In [20]:
#create a coloumn transformer for OHE
preprocessor=ColumnTransformer(
    transformers=[
        ("cat",OneHotEncoder(),categorical_features),
        ("num","passthrough",numerical_features)
    ]
)

In [21]:
#create a pipeline using preprocessing and random fprest classifier
pipeline=Pipeline(steps=[
    ('preprocessor',preprocessor),
    ("classifier",RandomForestClassifier(random_state=42))
])

In [22]:
X_train,X_test,y_train,y_test=train_test_split(X,y,random_state=1,test_size=0.3)
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [23]:
y_pred=pipeline.predict(X_test)

In [24]:
accuracy_score(y_test,y_pred)

0.7333333333333333

In [25]:
import pickle
#save the trained pipline using pickle
pickle_model_path="model.pkl"
with open(pickle_model_path,"wb")as f:
    pickle.dump(pipeline,f)